# 1 — Build the donor-partitioned cells parquet (DuckDB)

Streams the full QuPath `Cellmeasurements.csv` into
`data/cells/donor_id=*/*.parquet` (~59 marker means + morphology + region), using
DuckDB out-of-core (`PARTITION_BY donor_id`, ZSTD). This is the one already-parallel
stage — tune `duckdb_threads` in the **Parameters** cell below.

In [ ]:
# Parameters for this step (self-contained -- no config.ini). Edit the paths for your machine.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # make `phenocycler` importable from notebooks/
from phenocycler import PipelineConfig

REPO = pathlib.Path.cwd().resolve().parents[1]        # Islet-Explorer-Senior (parent of the submodule; data/ lives here)
# Run on a subset of donors (None = every donor under data/cells/donor_id=*).
DONORS = None            # e.g. ["6374", "6380"] to iterate on a few


cfg = PipelineConfig(
    data_dir  = REPO / "data",
    cells_csv = pathlib.Path("/home/smith6jt/IO60panc2nd/Cellmeasurements.csv"),
    # --- cells-parquet build (this step) ---
    duckdb_threads = 8,          # DuckDB threads for the out-of-core CSV -> parquet build
)
print("data_dir      :", cfg.data_dir)
print("cells_csv     :", cfg.cells_csv)
print("duckdb_threads:", cfg.duckdb_threads)
donors = DONORS or cfg.discover_donors()
print(f"donors: {len(donors)} " + ("(subset)" if DONORS else "(all)") + f" -> {donors[:6]}" + (" ..." if len(donors) > 6 else ""))


In [ ]:
# NOTE: the parquet build is ALL donors (one DuckDB COPY over the whole CSV); DONORS does not apply here.
from phenocycler.cells_parquet import build_cells_parquet

# Tip: pass limit=200_000 for a quick smoke test on a subset of rows.
build_cells_parquet(cfg, limit=None)

In [ ]:
# Confirm the partitions + donor list.
donors = cfg.discover_donors()
print(len(donors), 'donors:', donors)